In [41]:
N = 80 # number of qubits
#N = parse(Int, ARGS[1])

ndata = 1000 # number of data
nsamp = 500 # number of shots

tau = 1.0
gamma = 1.0

width = 2
exponent = 1
#width = parse(Int, ARGS[2])
#exponent = parse(Int, ARGS[3])

500

In [3]:
using SparseArrays
using LinearAlgebra
using KrylovKit
using Arpack
using Plots
using CSV
using DataFrames
using Printf
using PyCall
using ITensors, ITensorMPS
using ProgressBars
using Random
using JLD2
using Statistics

const pynp = pyimport("numpy")

PyObject <module 'numpy' from '/home/chinzei/.julia/conda/3/lib/python3.10/site-packages/numpy/__init__.py'>

In [4]:
BLAS.set_num_threads(1)
ITensors.blas_get_num_threads()

1

In [14]:
# original functions for shadow 
include("my_cs_tools_v250625.jl")

TESK (generic function with 1 method)

# Load data

In [45]:
AGL = [[j for j=i:i+width-1] for i=1:N-width+1] 

# training data
train_seed_list = [i for i=1:ndata]

train_cs = []
for seed in train_seed_list
    x = load("data_N=$(N)/cs_nsamp=$(nsamp)_seed=$(seed).jld2","cs")
    y = to_ClassicalShadow_TESK(x, gamma, tau, exponent, AGL)
    push!(train_cs, y)
end

# Kernel matrix

In [46]:
G = zeros(Float64, ndata, ndata)
for i=1:ndata
    for j=i:ndata
        cs1 = train_cs[i]
        cs2 = train_cs[j]
        G[i,j] = G[j,i] = TESK(cs1, cs2)
    end
end

# file output

In [47]:
filename = @sprintf("gram_matrix_N=%d/TESK_ndata=%d_nsamp=%d_w=%d_e=%d.csv", N, ndata, nsamp, width, exponent)
pynp.savetxt(filename, G, delimiter=",")